# BaseAgent - AlzKB

This notebook contains chronical records of BaseAgent's attempt in recreating AlzKB - v2.

In [1]:
%load_ext autoreload
%autoreload 2

import nest_asyncio
nest_asyncio.apply()

import os

from BaseAgent import BaseAgent, AgentTeam, MaxRoundsExceededError
from BaseAgent.agent_spec import AgentSpec


MCP_CONFIG = "examples/mcp_config.yaml"
SKILLS_DIR = "skills"
TEMPLATE_SRC = os.path.expanduser("~/GitHub/alzkb-updater")

Loaded environment variables from .env


In [2]:
def _print_token_summary(agents: list):
    """Print per-agent and total token counts from accumulated usage_metrics."""
    print("\n=== Token usage ===")
    totals = {"input": 0, "output": 0, "total": 0}
    for agent in agents:
        metrics = agent.usage_metrics
        input_tokens = sum(m.input_tokens or 0 for m in metrics)
        output_tokens = sum(m.output_tokens or 0 for m in metrics)
        total_tokens = sum(m.total_tokens or 0 for m in metrics)
        cost = sum(m.cost or 0.0 for m in metrics)
        cost_str = f"  ${cost:.4f}" if cost else ""
        print(f"  {agent.spec.name}: {input_tokens} in / {output_tokens} out / {total_tokens} total{cost_str}")
        totals["input"] += input_tokens
        totals["output"] += output_tokens
        totals["total"] += total_tokens
    print(f"  {'─' * 40}")
    print(f"  all agents:  {totals['input']} in / {totals['output']} out / {totals['total']} total")

# Run a KG-building agentic AI team

In [3]:
# ----- Ontology Agent -----
ontology_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="ontology_agent",
        role=(
            "A biomedical ontology engineer managing the OWL schema and project configuration. "
            "You own data/ontology/alzkb_v2.rdf and config/project.yaml: update disease_scope "
            "(primary_terms, umls_cuis, doid_ids, mesh_ids) and keep node_types/edge_types in "
            "sync with the RDF. Only modify the RDF on explicit request. Never edit Python source files."
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["ontology-protocol"],
    ),
    require_approval="never",
)
ontology_agent.add_mcp(MCP_CONFIG)

# ----- Database Agent -----
database_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="database_agent",
        role=(
            "A bioinformatics data engineer managing config/databases.yaml. "
            "You evaluate biomedical data sources and set enabled flags for the target disease. "
            "You do not write parsers or ontology mappings."
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["database-protocol"],
    ),
    require_approval="never",
)
database_agent.add_mcp(MCP_CONFIG)

# ----- Engineer Agent -----
engineer_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="engineer_agent",
        role=(
            "A Python software engineer writing parsers under src/parsers/. "
            "Each parser inherits from BaseParser and downloads data from one biomedical source, "
            "returning clean pandas DataFrames. Follow the full registration checklist: "
            "src/parsers/__init__.py, src/main.py PARSERS dict, test/eval_parser.py PARSER_CLASS_MAP. "
            "Run `python src/main.py --source <name>` to verify each parser produces TSVs. "
            "Never hardcode any disease-specific values in the parser code. "
            "Never hardcode any credentials."
            "You do not modify OWL files or ontology_mappings.yaml."
        ),
        llm="azure-claude-sonnet-4-6",
        skill_names=["parser-protocol"],
    ),
    require_approval="never",
)
engineer_agent.add_mcp(MCP_CONFIG)

# ----- Mapping Agent -----
mapping_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="mapping_agent",
        role=(
            "A knowledge graph mapping specialist owning config/ontology_mappings.yaml. "
            "You map processed TSV columns to OWL node types and relationship types. "
            "Other useful information such as gene descriptions, annotations, structures, and cross-references is included as properties. "
            "Always place node entries before relationship entries. "
            "Verify all OWL names against config/project.yaml node_types/edge_types before writing. "
            "Never edit Python source files."
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["mapping-protocol"],
    ),
    require_approval="never",
)
mapping_agent.add_mcp(MCP_CONFIG)

# ----- Memgraph Agent -----
memgraph_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="memgraph_agent",
        role=(
            "A graph database engineer who runs the full pipeline and validates graph export. "
            "Run `python src/main.py` inside the repo to produce data/output/alzkb_v2_populated.rdf, "
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["memgraph-protocol"],
    ),
    require_approval="never",
)
memgraph_agent.add_mcp(MCP_CONFIG)

# ----- Evaluator Agent -----
evaluator_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="evaluator_agent",
        role=(
            "A KG quality evaluator who runs eval_after_parser.py, eval_after_ontology.py, "
            "and eval_after_memgraph.py in sequence. Report tier-1 blocking failures "
            "(zero node/edge counts, OWL conformance < 1.0) and overall KG quality. "
            "Flag any blocking failures that must be resolved before the KG is used."
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["evaluation-protocol"],
    ),
    require_approval="never",
)
evaluator_agent.add_mcp(MCP_CONFIG)

# ----- HITL Agent -----
hitl_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="hitl_agent",
        role=(
            "A human review coordinator. Summarize the previous agent's output in less than 5 bullet "
            "points with descriptions on the task, target file(s), and rationale. "
            "Then call 'ask_user' with the summary and a clear yes/no question. "
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["hitl-protocol"],
    ),
    require_approval="never",
)
hitl_agent.add_mcp(MCP_CONFIG)


Auto-detecting source from model name: azure-claude-haiku-4-5
Creating AnthropicFoundry model: azure-claude-haiku-4-5
Skill 'ontology-protocol' successfully added

🔧 BASE AGENT CONFIGURATION
Agent Name: ontology_agent
LLM: azure-claude-haiku-4-5
Source: AnthropicFoundry
Base URL: None
Loaded Tools: ['run_python_repl', 'read_function_source_code']
Use Tool Retriever: False

Discovered 14 tools from filesystem MCP server (local)
Auto-detecting source from model name: azure-claude-haiku-4-5
Creating AnthropicFoundry model: azure-claude-haiku-4-5
Skill 'database-protocol' successfully added

🔧 BASE AGENT CONFIGURATION
Agent Name: database_agent
LLM: azure-claude-haiku-4-5
Source: AnthropicFoundry
Base URL: None
Loaded Tools: ['run_python_repl', 'read_function_source_code']
Use Tool Retriever: False

Discovered 14 tools from filesystem MCP server (local)
Auto-detecting source from model name: azure-claude-sonnet-4-6
Creating AnthropicFoundry model: azure-claude-sonnet-4-6
Skill 'parser-prot

In [ ]:

agents = [
    ontology_agent, database_agent, engineer_agent, evaluator_agent, mapping_agent, hitl_agent,
]

team = AgentTeam(
    agents=agents,
    supervisor_llm="azure-claude-opus-4-6",
    max_rounds=100,
)


TARGET_DATABASE = "gene ontology"
THREAD_ID = TARGET_DATABASE

task = f"""
Coordinate a team of agents to update and evaluate the {TARGET_DATABASE} parser in the repo at {TEMPLATE_SRC}. 

Available agents and their responsibilities:
- ontology_agent: Manage OWL schema in data/ontology/alzkb_v2.rdf and config/projects.yaml — ensure the project strictly conforms to the ontology.
- database_agent: Manage config/databases.yaml — enable or add data sources.
- engineer_agent: Manage parsers in src/parsers/. Verify with `python src/main.py --source <name>` run inside the repo.
- mapping_agent: Manage config/ontology_mappings.yaml — ensure the correct configurations (RDF classes, data property map, and merge/cross-reference).
- evaluator_agent: Focus on parser-specifc evaluation by running only eval_after_parser script and report the quality summary. List any tier-1 blocking failures explicitly.
- hitl_agent: Pause for user review after collecting all proposed changes to config files (config/*.yaml). Relay any feedback back to the responsible agent.


Instructions:
- Find more information about the {TARGET_DATABASE} data and the download file contents from the official {TARGET_DATABASE} documentation.
- Include useful information as properties. For drugs, useful information includes the drug description, index, structure (in SMILES and InChI formats), FDA approval status, and more. For diseases, useful information include disease descriptions, symptoms, and other relevant details. For genes, useful information includes expression, call quality, FDR, expression score, and expression rank.
- Coordinate across agents to produce a verified {TARGET_DATABASE} parser with no tier-1 blocking failures.
- Add new RDF classes when and only when the extracted data can be cross referenced to other sources using standardized identifiers. Free-text descriptions can only be included as properties, not RDF classes.
- Upon user approval, update OWL schema in data/ontology/alzkb_v2.rdf and graph schema in config/ontology_mappings.yaml to accommodate the new RDF classes.
- Upon user approval, update "data_property_map" in config/ontology_mappings.yaml to accommodate the new RDF properties.


Inter-agent contracts:
- All parsers must be verified before evaluator_agent runs.
- mapping_agent must report valid cross-reference information before the engineer_agent starts.
- On tier-1 failures, route back to engineer_agent to fix before re-evaluating.


Strict contracts and constraints:
- Do not include any filter (e.g., disease, tissue, or drug) if the specific parser doesn't include these RDF classes.
- Directly retrieve data from the source database. Do not use any precomputed third-party processed data. Do not use any precomputed data from Hetionet.
- config/project.yaml sets project.name, display_name, and all disease_scope fields (primary_terms, umls_cuis, doid_ids, mesh_ids). 
- The databases.yaml key, PARSERS key, PARSER_CLASS_MAP key, ontology_mappings.yaml prefix, and data/processed/ subdirectory name must all be identical strings.
- In ontology_mappings.yaml, all node entries must precede all relationship entries.
- In ontology_mappings.yaml, merge must be true is the
- OWL names in ontology_mappings.yaml must be active (uncommented) in project.yaml.
- Credentials use the _env suffix in databases.yaml args; the parser constructor must accept the stripped parameter name.
"""

Auto-detecting source from model name: azure-claude-opus-4-6
Creating AnthropicFoundry model: azure-claude-opus-4-6


In [5]:
_log, result = team.run_sync(task, thread_id=THREAD_ID)


[supervisor] → mapping_agent: We need to update/create a gene ontology parser in the repo at /Users/lib/GitHub/alzkb-updater. Before the engineer can start, we need you to investigate and report the following:

1. Read the current config/ontology_mappings.yaml and identify any existing Gene Ontology (GO) related entries (nodes, relationships, properties).
2. Read config/project.yaml to understand the current node_types and edge_types that are active.
3. Research the Gene Ontology (GO) official data source (http://geneontology.org/). The GO provides:
   - Gene Ontology term definitions (GO terms) with GO IDs (e.g., GO:0008150)
   - Gene annotations linking genes to GO terms (GAF files from http://current.geneontology.org/annotations/)
   - Three sub-ontologies: Biological Process, Molecular Function, Cellular Component
   - Cross-references to UniProt, NCBI Gene, etc.

4. Determine what valid cross-reference identifiers are available from GO data:
   - GO term IDs (GO:XXXXXXX) for onto

<string>:2: SyntaxWarning: invalid escape sequence '\|'


================================== Ai Message ==================================

<think>
Great! Now let me look at the actual class definitions to understand what data properties they have.
</think>
<execute>
# Get the class definitions for BiologicalProcess, MolecularFunction, and CellularComponent
result = subprocess.run(['sed', '-n', '2086,2150p', '/Users/lib/GitHub/alzkb-updater/data/ontology/alzkb_v2.rdf'], 
                       capture_output=True, text=True)
print("=== BiologicalProcess class definition ===")
print(result.stdout)

result = subprocess.run(['sed', '-n', '2150,2210p', '/Users/lib/GitHub/alzkb-updater/data/ontology/alzkb_v2.rdf'], 
                       capture_output=True, text=True)
print("\n=== CellularComponent class definition ===")
print(result.stdout)

result = subprocess.run(['sed', '-n', '2700,2760p', '/Users/lib/GitHub/alzkb-updater/data/ontology/alzkb_v2.rdf'], 
                       capture_output=True, text=True)
print("\n=== MolecularFunction clas

<string>:2: SyntaxWarning: invalid escape sequence '\|'


================================== Ai Message ==================================

<think>
Let me search more specifically for GO-related properties or if they need to be created. Let me also look for what properties already exist for BiologicalProcess, MolecularFunction, and CellularComponent.
</thinking>
<execute>
# Search for all data properties in the RDF file to understand the pattern
result = subprocess.run(['grep', '-E', 'DatatypeProperty|xref', 
                        '/Users/lib/GitHub/alzkb-updater/data/ontology/alzkb_v2.rdf'], 
                       capture_output=True, text=True)
lines = result.stdout.split('\n')
# Filter for unique xref properties
xref_props = set()
for line in lines:
    if 'xref' in line and 'DatatypeProperty' in line:
        # Extract the property name
        if 'rdf:about=' in line:
            start = line.find('#') + 1
            end = line.find('"', start)
            if start > 0 and end > start:
                xref_props.add(line[start:end])


<string>:2: SyntaxWarning: invalid escape sequence '\|'
<string>:19: SyntaxWarning: invalid escape sequence '\|'


================================== Ai Message ==================================

<think>
Perfect! Now I have all the information I need to create a comprehensive investigation report and propose the ontology_mappings.yaml structure for the gene_ontology parser. Let me create a detailed report with all findings and proposed mappings.
</think>
<execute>
# Now let's create a comprehensive investigation report
# First, let me verify one more thing - the current state of ontology_mappings.yaml to see if there are any GO entries already
result = subprocess.run(['grep', '-n', 'gene_ontology', '/Users/lib/GitHub/alzkb-updater/config/ontology_mappings.yaml'], 
                       capture_output=True, text=True)
print("=== Current GO entries in ontology_mappings.yaml ===")
if result.stdout:
    print(result.stdout)
else:
    print("No gene_ontology entries found in ontology_mappings.yaml")

# Let's also check the end of the file to see where we should add new entries
result = subprocess.run(

In [ ]:
_log, result = team.run_sync("Continue", thread_id=THREAD_ID)


[supervisor] → FINISH


# Sub-agent follow-up

In [22]:
team_tid = team._current_thread_id
_log2, result2 = evaluator_agent.run(
    f"Report the evaluation status of the {TARGET_DATABASE} parser",
    thread_id=f"{team_tid}:evaluator_agent"
)

================================ Human Message =================================

Report the evaluation status of the disgenet parser
================================== Ai Message ==================================

<think>
The user is asking me to report the evaluation status of the disgenet parser. Based on the skill provided (evaluation-protocol), I need to:

1. First, understand the structure of the project and locate the evaluation scripts
2. Run the eval_after_parser.py script to check the disgenet parser specifically
3. Report on the results, particularly focusing on:
   - Tier 1 blocking failures (zero node/edge counts, OWL conformance < 1.0)
   - Overall KG quality metrics
   - Any issues specific to disgenet

Let me start by:
1. Checking the allowed directories
2. Exploring the project structure
3. Locating and running the eval_after_parser.py script
4. Parsing and interpreting the results
</think>

I'll help you evaluate the disgenet parser. Let me start by exploring the pro

# Token usage summary

In [12]:
_print_token_summary(agents)


=== Token usage ===
  ontology_agent: 649274 in / 47957 out / 697231 total
  database_agent: 0 in / 0 out / 0 total
  engineer_agent: 2587181 in / 48521 out / 2635702 total
  evaluator_agent: 4366530 in / 175777 out / 4542307 total
  mapping_agent: 7072170 in / 283313 out / 7355483 total
  hitl_agent: 1230897 in / 146816 out / 1377713 total
  ────────────────────────────────────────
  all agents:  15906052 in / 702384 out / 16608436 total
